# Insightia — Notebook 04 (V2) : Bloc 4 — Tendances & signaux faibles

**Objectif** : anticiper plutôt que subir.

Ce notebook produit :
- l'évolution **mensuelle** des **motifs**
- l'évolution **mensuelle** des **situations** (issues des règles Notebook 03)
- des **signaux faibles** : faible volume mais forte hausse récente
- un **zoom** : couples *(motif × situation)* qui montent

**Question** : *« Qu’est-ce qui est en train de devenir un problème ? »*

## Fichiers générés (outputs/)
- `block4_motif_month.csv`
- `block4_situation_month.csv`
- `block4_signals_motif.csv`
- `block4_signals_situation.csv`
- `block4_zoom_motif_situation.csv`
- `block4_summary.json`


## 1) Imports + paramètres

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PARQUET = OUT_DIR / "comments_clean.parquet"


## 2) Chargement data (parquet enrichi)

In [ ]:

if not DATA_PARQUET.exists():
    raise FileNotFoundError("outputs/comments_clean.parquet introuvable. Exécute Notebook 02 puis Notebook 03 (rewrite).")

df = pd.read_parquet(DATA_PARQUET)

print("Nb lignes:", len(df))
print("Colonnes:", list(df.columns))
display(df.head(3))


## 3) Création robuste de `month` (corrige KeyError: 'month')

In [ ]:

if "date" not in df.columns:
    raise KeyError("Colonne `date` absente. Le Bloc 4 nécessite une date.")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).copy()

# Début du mois (standard pour séries temporelles)
df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()

print("Période:", df["date"].min().date(), "→", df["date"].max().date())
display(df[["date","month","motif","sentiment"]].head(5))


## 4) Normalisation robuste de `situations` (corrige ValueError pd.isna(array) )

In [ ]:

if "situations" not in df.columns:
    raise KeyError("Colonne `situations` absente. Exécute Notebook 03 (rewrite) pour la reconstruire.")

def ensure_list(x):
    # Cas 1 : déjà une liste Python
    if isinstance(x, list):
        return x

    # Cas 2 : None / NaN
    if x is None:
        return []
    # NaN float (le plus fréquent)
    if isinstance(x, float) and pd.isna(x):
        return []

    # Cas 3 : numpy array / pandas Series (évite l'ambiguïté pd.isna(array))
    if isinstance(x, (np.ndarray, pd.Series)):
        return list(x)

    # Cas 4 : string "a,b,c"
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return []
        return [t.strip() for t in x.split(",") if t.strip()]

    # Fallback
    return []

df["situations"] = df["situations"].apply(ensure_list)

# Contrôle types
type_counts = df["situations"].apply(type).value_counts()
print("Types dans `situations` :")
display(type_counts)

share = (df["situations"].apply(len) > 0).mean()
print("Part commentaires avec ≥1 situation :", round(float(share), 3))

display(df[["motif","sentiment","situations","commentaire"]].head(5))


## 5) Série temporelle : motifs par mois

In [ ]:

motif_month = (
    df.groupby(["month","motif"])
      .size()
      .reset_index(name="n")
      .sort_values(["month","n"], ascending=[True, False])
)

motif_month.to_csv(OUT_DIR / "block4_motif_month.csv", index=False)
print("✅ Export:", OUT_DIR / "block4_motif_month.csv")
display(motif_month.head(15))


## 6) Série temporelle : situations par mois (format long)

In [ ]:

rows = []
for _, r in df.iterrows():
    sits = r["situations"]
    for s in sits:
        rows.append({"month": r["month"], "motif": r["motif"], "situation": s})

df_sit = pd.DataFrame(rows)

if df_sit.empty:
    raise ValueError("Aucune situation détectée (df_sit vide). Vérifie les règles Notebook 03 / la colonne situations.")

situation_month = (
    df_sit.groupby(["month","situation"])
          .size()
          .reset_index(name="n")
          .sort_values(["month","n"], ascending=[True, False])
)

situation_month.to_csv(OUT_DIR / "block4_situation_month.csv", index=False)
print("✅ Export:", OUT_DIR / "block4_situation_month.csv")
display(situation_month.head(15))


## 7) Détection de signaux faibles (fenêtre récente vs précédente)

In [ ]:

def detect_signals(ts_df, key_col, n_recent_months=3, min_recent=20):
    """Compare somme récente vs somme précédente (fenêtres de même taille)."""
    ts_df = ts_df.copy()
    ts_df["month"] = pd.to_datetime(ts_df["month"])

    months = sorted(ts_df["month"].unique())
    if len(months) < 2:
        raise ValueError("Pas assez de mois pour calculer une tendance.")
    if len(months) < 2 * n_recent_months:
        n_recent_months = max(1, len(months)//2)

    recent = months[-n_recent_months:]
    prev = months[-2*n_recent_months:-n_recent_months]

    agg_recent = ts_df[ts_df["month"].isin(recent)].groupby(key_col)["n"].sum().rename("n_recent")
    agg_prev   = ts_df[ts_df["month"].isin(prev)].groupby(key_col)["n"].sum().rename("n_prev")

    out = pd.concat([agg_recent, agg_prev], axis=1).fillna(0)
    out["delta"] = out["n_recent"] - out["n_prev"]
    out["growth_ratio"] = (out["n_recent"] + 1) / (out["n_prev"] + 1)

    out = out.reset_index()
    out = out[out["n_recent"] >= min_recent].copy()
    out = out.sort_values(["growth_ratio","delta","n_recent"], ascending=[False, False, False])

    out["recent_months"] = ", ".join([str(pd.to_datetime(m).date()) for m in recent])
    out["prev_months"] = ", ".join([str(pd.to_datetime(m).date()) for m in prev]) if len(prev) else ""
    return out


## 8) Signaux faibles : motifs

In [ ]:

sig_motif = detect_signals(motif_month, key_col="motif", n_recent_months=3, min_recent=30)
sig_motif.to_csv(OUT_DIR / "block4_signals_motif.csv", index=False)
print("✅ Export:", OUT_DIR / "block4_signals_motif.csv")
display(sig_motif.head(20))


## 9) Signaux faibles : situations

In [ ]:

sig_situation = detect_signals(situation_month, key_col="situation", n_recent_months=3, min_recent=20)
sig_situation.to_csv(OUT_DIR / "block4_signals_situation.csv", index=False)
print("✅ Export:", OUT_DIR / "block4_signals_situation.csv")
display(sig_situation.head(20))


## 10) Zoom : couples (motif × situation) qui montent

In [ ]:

motif_sit_month = (
    df_sit.groupby(["month","motif","situation"])
          .size()
          .reset_index(name="n")
)

def detect_signals_pair(ts_df, n_recent_months=3, min_recent=20):
    ts_df = ts_df.copy()
    ts_df["month"] = pd.to_datetime(ts_df["month"])

    months = sorted(ts_df["month"].unique())
    if len(months) < 2:
        raise ValueError("Pas assez de mois pour calculer une tendance.")
    if len(months) < 2 * n_recent_months:
        n_recent_months = max(1, len(months)//2)

    recent = months[-n_recent_months:]
    prev = months[-2*n_recent_months:-n_recent_months]

    key_cols = ["motif","situation"]
    agg_recent = ts_df[ts_df["month"].isin(recent)].groupby(key_cols)["n"].sum().rename("n_recent")
    agg_prev   = ts_df[ts_df["month"].isin(prev)].groupby(key_cols)["n"].sum().rename("n_prev")

    out = pd.concat([agg_recent, agg_prev], axis=1).fillna(0)
    out["delta"] = out["n_recent"] - out["n_prev"]
    out["growth_ratio"] = (out["n_recent"] + 1) / (out["n_prev"] + 1)

    out = out.reset_index()
    out = out[out["n_recent"] >= min_recent].copy()
    out = out.sort_values(["growth_ratio","delta","n_recent"], ascending=[False, False, False])

    out["recent_months"] = ", ".join([str(pd.to_datetime(m).date()) for m in recent])
    out["prev_months"] = ", ".join([str(pd.to_datetime(m).date()) for m in prev]) if len(prev) else ""
    return out

zoom_pair = detect_signals_pair(motif_sit_month, n_recent_months=3, min_recent=20)
zoom_pair.to_csv(OUT_DIR / "block4_zoom_motif_situation.csv", index=False)
print("✅ Export:", OUT_DIR / "block4_zoom_motif_situation.csv")
display(zoom_pair.head(25))


## 11) Résumé & traçabilité

In [ ]:

summary = {
    "n_rows": int(len(df)),
    "period_min": str(df["date"].min().date()),
    "period_max": str(df["date"].max().date()),
    "share_with_situation": float((df["situations"].apply(len) > 0).mean()),
    "outputs": {
        "motif_month": "outputs/block4_motif_month.csv",
        "situation_month": "outputs/block4_situation_month.csv",
        "signals_motif": "outputs/block4_signals_motif.csv",
        "signals_situation": "outputs/block4_signals_situation.csv",
        "zoom_motif_situation": "outputs/block4_zoom_motif_situation.csv",
    }
}

with open(OUT_DIR / "block4_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("✅ Export:", OUT_DIR / "block4_summary.json")
OUT_DIR / "block4_summary.json"


✅ Fin Notebook 04 (V2).

**Next : Notebook 05 — Backlog**
On combinera :
- volume motif (Bloc 2)
- part négative / note (Bloc 2)
- tendance / signaux faibles (Bloc 4)
- hotspots contextuels (Bloc 3)
